In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import udf
from pyspark.sql.types import StringType

spark = SparkSession.builder.appName("UDF").getOrCreate()

@udf(returnType=StringType())
def user_deets(name, age):
    return f"Name: {name}, Age: {age}"

peeps = spark.read.csv("../data/fakefriends-header.csv", header=True)

peeps.show()

peeps.withColumn("name_age", user_deets("name", "age")).show()
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/02 13:02:49 WARN Utils: Your hostname, Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/02 13:02:49 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 13:02:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+------+--------+---+-------+
|userID|    name|age|friends|
+------+--------+---+-------+
|     0|    Will| 33|    385|
|     1|Jean-Luc| 26|      2|
|     2|    Hugh| 55|    221|
|     3|  Deanna| 40|    465|
|     4|   Quark| 68|     21|
|     5|  Weyoun| 59|    318|
|     6|  Gowron| 37|    220|
|     7|    Will| 54|    307|
|     8|  Jadzia| 38|    380|
|     9|    Hugh| 27|    181|
|    10|     Odo| 53|    191|
|    11|     Ben| 57|    372|
|    12|   Keiko| 54|    253|
|    13|Jean-Luc| 56|    444|
|    14|    Hugh| 43|     49|
|    15|     Rom| 36|     49|
|    16|  Weyoun| 22|    323|
|    17|     Odo| 35|     13|
|    18|Jean-Luc| 45|    455|
|    19|  Geordi| 60|    246|
+------+--------+---+-------+
only showing top 20 rows


+------+--------+---+-------+--------------------+
|userID|    name|age|friends|            name_age|
+------+--------+---+-------+--------------------+
|     0|    Will| 33|    385| Name: Will, Age: 33|
|     1|Jean-Luc| 26|      2|Name: Jean-Luc, A...|
|     2|    Hugh| 55|    221| Name: Hugh, Age: 55|
|     3|  Deanna| 40|    465|Name: Deanna, Age...|
|     4|   Quark| 68|     21|Name: Quark, Age: 68|
|     5|  Weyoun| 59|    318|Name: Weyoun, Age...|
|     6|  Gowron| 37|    220|Name: Gowron, Age...|
|     7|    Will| 54|    307| Name: Will, Age: 54|
|     8|  Jadzia| 38|    380|Name: Jadzia, Age...|
|     9|    Hugh| 27|    181| Name: Hugh, Age: 27|
|    10|     Odo| 53|    191|  Name: Odo, Age: 53|
|    11|     Ben| 57|    372|  Name: Ben, Age: 57|
|    12|   Keiko| 54|    253|Name: Keiko, Age: 54|
|    13|Jean-Luc| 56|    444|Name: Jean-Luc, A...|
|    14|    Hugh| 43|     49| Name: Hugh, Age: 43|
|    15|     Rom| 36|     49|  Name: Rom, Age: 36|
|    16|  Weyoun| 22|    323|Na

In [21]:
from pyspark.sql import SparkSession, functions as func

spark = SparkSession.builder.appName("hashtagAggregating").getOrCreate()

inputDF = spark.read.text('../data/pride_and_prejudice.txt')

words = inputDF.select(func.explode(func.split(inputDF.value, r'\W+')).alias('word'))
words_no_empty = words.filter(words.word != '')
lowercase_words = words_no_empty.select(func.lower(words_no_empty.word).alias('word'))
word_counts = lowercase_words.groupBy('word').count().sort('count', ascending=False)
word_counts.show()

spark.stop()


+----+-----+
|word|count|
+----+-----+
| the| 4846|
|  to| 4405|
|  of| 3962|
| and| 3835|
| her| 2260|
|   i| 2098|
|   a| 2094|
|  in| 2051|
| was| 1874|
| she| 1732|
|that| 1620|
|  it| 1603|
| not| 1520|
| you| 1417|
|  he| 1350|
| his| 1289|
|  be| 1280|
|  as| 1239|
| had| 1181|
|with| 1149|
+----+-----+
only showing top 20 rows


In [26]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType, DatetimeType

spark = SparkSession.builder.appName("Schema Time").getOrCreate()

schema = StructType([
    StructField('StationId', StringType()),
    StructField('date', IntegerType()),
    StructField('type', StringType()),
    StructField('value', FloatType())
])

df = spark.read.csv(
    '../data/weatherData1800s.csv',
    schema=schema,
    header=False
)

df.printSchema()
df.show()
min_temps = df.filter(df.type == 'TMIN')
station_temps = min_temps.select('StationID', 'value')
min_temp_by_station = station_temps.groupBy('StationID').min('value').show()





spark.stop()

root
 |-- StationId: string (nullable = true)
 |-- date: integer (nullable = true)
 |-- type: string (nullable = true)
 |-- value: float (nullable = true)

+-----------+--------+----+------+
|  StationId|    date|type| value|
+-----------+--------+----+------+
|ITE00100554|18000101|TMAX| -75.0|
|ITE00100554|18000101|TMIN|-148.0|
|GM000010962|18000101|PRCP|   0.0|
|EZE00100082|18000101|TMAX| -86.0|
|EZE00100082|18000101|TMIN|-135.0|
|ITE00100554|18000102|TMAX| -60.0|
|ITE00100554|18000102|TMIN|-125.0|
|GM000010962|18000102|PRCP|   0.0|
|EZE00100082|18000102|TMAX| -44.0|
|EZE00100082|18000102|TMIN|-130.0|
|ITE00100554|18000103|TMAX| -23.0|
|ITE00100554|18000103|TMIN| -46.0|
|GM000010962|18000103|PRCP|   4.0|
|EZE00100082|18000103|TMAX| -10.0|
|EZE00100082|18000103|TMIN| -73.0|
|ITE00100554|18000104|TMAX|   0.0|
|ITE00100554|18000104|TMIN| -13.0|
|GM000010962|18000104|PRCP|   0.0|
|EZE00100082|18000104|TMAX| -55.0|
|EZE00100082|18000104|TMIN| -74.0|
+-----------+--------+----+------+
only

In [1]:
from pyspark.sql import SparkSession, functions as func
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, FloatType

spark = SparkSession.builder.appName("Order Totals").getOrCreate()

schema = StructType([
    StructField('cust_id', IntegerType()),
    StructField('order_id', IntegerType()),
    StructField('cost', FloatType())
])

df = spark.read.csv(
    "../data/customer-orders.csv",
    header=False,
    schema=schema
)

only_custies = df.select("cust_id", "cost")
total_spending = only_custies.groupBy("cust_id").agg(func.round(func.sum("cost"), 2).alias("total_cost")).sort("total_cost", ascending=False)
total_spending.show()

totals = spark.sql

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/02 11:18:25 WARN Utils: Your hostname, Laptop, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/02 11:18:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/02 11:18:26 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+-------+----------+
|cust_id|total_cost|
+-------+----------+
|     68|   6375.45|
|     73|    6206.2|
|     39|   6193.11|
|     54|   6065.39|
|     71|   5995.66|
|      2|   5994.59|
|     97|   5977.19|
|     46|   5963.11|
|     42|   5696.84|
|     59|   5642.89|
|     41|   5637.62|
|      0|   5524.95|
|      8|   5517.24|
|     85|   5503.43|
|     61|   5497.48|
|     32|   5496.05|
|     58|   5437.73|
|     63|   5415.15|
|     15|   5413.51|
|      6|   5397.88|
+-------+----------+
only showing top 20 rows


In [10]:
from pyspark.sql import SparkSession
import os
import pyspark.pandas as ps


os.environ['PYARROW_IGNORE_TIMEZONE'] = "1"


spark = SparkSession.builder \
    .appName("Sparky Panda") \
    .config("spark.sql.ansi.enable", "false") \
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

ps_df = ps.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bobby", "David", "Charlie", "Jimmy"],
    "age": [25, 30, 28, 96, 32],
    "salary": [50000, 60000, 87000, 123000, 64000]
})

# print(ps_df)

print("\nAvg Age: ", ps_df["age"].mean())
print("\nSummary Stats\n ", ps_df.describe())

ps_df['bonus'] = ps_df['salary'] * 0.1
print("\nNow with bonus\n", ps_df)

filtered = ps_df[ps_df["age"] > 30]
print("\nfiltered\n", ps_df)

spark_df = ps_df.to_spark()
# spark_df = spark.createDataFrame(ps_df)
spark_df.printSchema()

print("\nSpark DF\n")
spark_df.show()

ps_df_from_spark = ps.DataFrame(spark_df)
print("\nfrom spark df \n")
spark.stop()


Avg Age:  42.2

Summary Stats
               id       age         salary
count  5.000000   5.00000       5.000000
mean   3.000000  42.20000   76800.000000
std    1.581139  30.18609   29166.761905
min    1.000000  25.00000   50000.000000
25%    2.000000  28.00000   60000.000000
50%    3.000000  30.00000   64000.000000
75%    4.000000  32.00000   87000.000000
max    5.000000  96.00000  123000.000000

Now with bonus
    id     name  age  salary    bonus
0   1    Alice   25   50000   5000.0
1   2    Bobby   30   60000   6000.0
2   3    David   28   87000   8700.0
3   4  Charlie   96  123000  12300.0
4   5    Jimmy   32   64000   6400.0

filtered
    id     name  age  salary    bonus
0   1    Alice   25   50000   5000.0
1   2    Bobby   30   60000   6000.0
2   3    David   28   87000   8700.0
3   4  Charlie   96  123000  12300.0
4   5    Jimmy   32   64000   6400.0
root
 |-- id: long (nullable = false)
 |-- name: string (nullable = false)
 |-- age: long (nullable = false)
 |-- salary: long

/home/mikek444/project/spark/.venv/lib/python3.12/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If `index_col` is not specified for `to_spark`, the existing index is lost when converting to Spark DataFrame.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


+---+-------+---+------+-------+
| id|   name|age|salary|  bonus|
+---+-------+---+------+-------+
|  1|  Alice| 25| 50000| 5000.0|
|  2|  Bobby| 30| 60000| 6000.0|
|  3|  David| 28| 87000| 8700.0|
|  4|Charlie| 96|123000|12300.0|
|  5|  Jimmy| 32| 64000| 6400.0|
+---+-------+---+------+-------+


from spark df 



In [16]:
from pyspark.sql import SparkSession
import os
import pyspark.pandas as ps


os.environ['PYARROW_IGNORE_TIMEZONE'] = "1"


spark = SparkSession.builder \
    .appName("Sparky Panda") \
    .config("spark.sql.ansi.enable", "false") \
    .config("spark.executorEnv.PYARROW_IGNORE_TIMEZONE", "1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

ps_df = ps.DataFrame({
    "id": [1,2,3,4,5],
    "name": ["Alice", "Bobby", "David", "Charlie", "Jimmy"],
    "age": [25, 30, 28, 96, 32],
    "salary": [50000, 60000, 87000, 123000, 64000]
})



# using transform for element-wise operations
ps_df["age_in_10_years"] = ps_df["age"].transform(lambda x: x + 10)

# using apply(): function that will run for each value in a column

def categorize(age):
    if age < 30:
        return  "Spring Chicken"
    elif age < 65:
        return "Senior Dev"
    else:
        return "DEAD"
    
ps_df["Age Bracket"] = ps_df["age"].apply(categorize)

def format_row(row):
    return f"{row["name"]} is {row["age"]} years old"
ps_df["name_with_age"] = ps_df.apply(format_row, axis = 1)
print(ps_df)

spark.stop()

/home/mikek444/project/spark/.venv/lib/python3.12/site-packages/pyspark/pandas/utils.py:1038: PandasAPIOnSparkAdviceWarning: If the type hints is not specified for `apply`, it is expensive to infer the data type internally.
  warnings.warn(message, PandasAPIOnSparkAdviceWarning)


   id     name  age  salary  age_in_10_years     Age Bracket            name_with_age
0   1    Alice   25   50000               35  Spring Chicken    Alice is 25 years old
1   2    Bobby   30   60000               40      Senior Dev    Bobby is 30 years old
2   3    David   28   87000               38  Spring Chicken    David is 28 years old
3   4  Charlie   96  123000              106            DEAD  Charlie is 96 years old
4   5    Jimmy   32   64000               42      Senior Dev    Jimmy is 32 years old
